### nb_01 — 着地ファイル → ブロンズ層テーブル

**アタッチするレイクハウス**：`lh_its_asset_bronze`

**前提**：`data` 配下のソースシステム別フォルダーを、`lh_its_asset_bronze` の
`Files/` にフォルダーごとアップロード済みであること。

| | |
|---|---|
| 入力 | `Files/<system>/<entity>/<yyyy>/<MM>/<dd>/<entity>_YYYYMMDD.csv` |
| 出力 | `lh_its_asset_bronze` の `Tables/bz_<system>_<entity>` |

#### このノートブックの役割

ブロンズ層は「取り込んだものを、取り込んだまま、すべて残す」層です。取り込み日が
複数あれば、全日分をそのまま保持します。重複排除も最新版の選別も一切行いません。
それはシルバー層（`nb_02`）の責務です。

したがって日を重ねるとブロンズ層の行数は増え続けます。**それが正しい状態です。**
「8月5日時点ではこう見えていた」を後から再現できることが、この層の価値です。

#### 着地ファイルとテーブルを一致させる

このノートブックは、実行のたびに **`Files/` の内容とブロンズ層のテーブル一覧を
一致させます**。

| 状態 | 動作 |
|---|---|
| CSV があり、テーブルが無い | テーブルを作る |
| CSV があり、テーブルもある | 全日分を読み直して置き換える |
| **CSV が無く、テーブルだけある** | **テーブルを削除する** |

3行目が必要なのは、ソースシステムの構成を変えたときです。取り込み対象から外した
エンティティのテーブルは、上書きの対象にならないため放っておくと残り続けます。
実体のない CSV に対応するテーブルがブロンズ層に並ぶと、後続の層から見て
「あるはずのデータが無い」「無いはずのデータがある」の判別がつかなくなります。

削除は `SOURCES` に書いた接頭辞（`bz_`）を持つテーブルだけが対象です。別用途で
置いたテーブルは触りません。確認だけしたい場合は `DROP_ORPHAN_TABLES = False`
にすると、対象を表示するだけで削除しません。

#### 設定

取り込むソースシステムとエンティティの一覧です。

In [ ]:
from pyspark.sql import functions as F
import notebookutils

FILES_ROOT = "Files"

# 定義から外れたブロンズ層テーブル（対応する CSV が無いもの）を削除するか。
# False にすると、対象を表示するだけで削除しません。
DROP_ORPHAN_TABLES = True

# ソースシステムと取り込み対象エンティティ
SOURCES = {
    # Entra ID は「人と組織の権威」ではない。公式の推奨構成では、SAP SuccessFactors の
    # ような人事システムが従業員の記録システムであり、そこから Entra ID へ HR 主導で
    # プロビジョニングされる。氏名・部署は人事に由来し、Entra ID はその配信先。
    # したがって Entra ID からは、そこに実在する属性だけを取り込む。
    "entra_id": ["person"],                                             # Microsoft Entra ID
    "successfactors": ["employee_profile", "organization",              # SAP SuccessFactors
                       "person_skill", "person_certification"],
    "d365_project_operations": ["availability", "assignment"],          # D365 Project Operations
    "servicenow": ["project", "project_technology", "project_required_skill"],  # ServiceNow
    "salesforce": ["customer", "stakeholder", "opportunity"],           # Salesforce Sales Cloud
    "sharepoint_online": ["deliverable"],                               # SharePoint Online
    "confluence": ["knowledge"],                                        # Atlassian Confluence
    "dataverse": ["skill", "certification", "technology"],              # Microsoft Dataverse
}

# 着地パスの日付階層 .../<yyyy>/<MM>/<dd>/... から取り込み日を復元する。
# Hive スタイル（ingest_date=YYYY-MM-DD）ではないため Spark の自動認識は効かないが、
# パイプラインが着地させるランディングゾーンは年/月/日の階層であることが多く、
# 実運用でもこのようにパスから日付を導出するのが一般的。
DATE_FROM_PATH = r".*/(\d{4})/(\d{2})/(\d{2})/[^/]+$"

#### 取り込む

各エンティティのフォルダーを再帰的に読み、着地パスの日付階層から `ingest_date` を
復元してからブロンズ層のテーブルに書き出します。全日分を読み直して丸ごと置き換える
ため、何度実行しても結果は同じになります。

In [ ]:
results = []
skipped = []


def has_files(path):
    """着地パスに実ファイルが1つでもあるか（空フォルダーは「無い」とみなす）。"""
    stack = [path]
    while stack:
        try:
            entries = notebookutils.fs.ls(stack.pop())
        except Exception:
            continue
        for e in entries:
            if e.isDir:
                stack.append(e.path)
            elif e.size > 0:
                return True
    return False


for src, entities in SOURCES.items():
    for entity in entities:
        path = f"{FILES_ROOT}/{src}/{entity}"

        # CSV が無いエンティティは飛ばす。テーブルは後段の整理で削除される。
        if not has_files(path):
            skipped.append(f"{src}/{entity}")
            continue

        df = (
            spark.read.option("header", "true")
            .option("inferSchema", "true")
            .option("encoding", "UTF-8")
            .option("recursiveFileLookup", "true")  # 年/月/日の階層を再帰的に読む
            .csv(path)
        )
        # 列名の BOM と前後空白を除去
        for c in df.columns:
            df = df.withColumnRenamed(c, c.strip().replace("\ufeff", ""))

        # ファイルパスから ingest_date を復元（シルバー層で最新判定に使う重要な列）
        src_path = F.input_file_name()
        df = df.withColumn(
            "ingest_date",
            F.concat_ws(
                "-",
                F.regexp_extract(src_path, DATE_FROM_PATH, 1),
                F.regexp_extract(src_path, DATE_FROM_PATH, 2),
                F.regexp_extract(src_path, DATE_FROM_PATH, 3),
            ),
        )

        bad_path = df.filter(F.col("ingest_date").rlike(r"^-|-$|^$")).count()
        if bad_path > 0:
            print(
                f"[WARN] {src}/{entity}: 日付を判定できないファイルが {bad_path} 件あります。"
                f" Files/{src}/{entity}/<yyyy>/<MM>/<dd>/ の階層になっているか確認してください。"
            )

        # ブロンズ層の監査列。ingest_date は絶対に落とさない（シルバー層で使う）
        df = (
            df.withColumn("_source_system", F.lit(src))
              .withColumn("_source_entity", F.lit(entity))
              .withColumn("_source_file", F.element_at(F.split(src_path, "/"), -1))
              .withColumn("_ingested_at", F.current_timestamp())
        )

        table = f"bz_{src}_{entity}"
        # 全日分を読み直して丸ごと置き換えるため、何度実行しても結果は同じ
        df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table)

        n = df.count()
        dates = sorted(r["ingest_date"] for r in df.select("ingest_date").distinct().collect())
        results.append((table, n, len(dates), ", ".join(dates)))
        print(f"{table}: {n} rows / {len(dates)} date(s) [{', '.join(dates)}]")

print(f"\n{len(results)} tables written to lh_its_asset_bronze")

if skipped:
    print(f"\n[SKIP] 着地ファイルが無いエンティティ ({len(skipped)}): {', '.join(skipped)}")
    print("       Files/ にアップロードされていないか、取り込み対象から外れています。")


# ============================================================
# 着地ファイルとテーブル一覧を一致させる
# ============================================================
# saveAsTable(overwrite) は「書いたテーブル」しか触りません。取り込み対象から
# 外したエンティティのテーブルは上書きされないため、放っておくと残り続けます。
# CSV が存在しないのにテーブルだけがある状態は、後続の層から見て紛らわしいので、
# ここで実体に合わせます。
def bronze_table_names():
    try:
        return [t.name for t in spark.catalog.listTables()]
    except Exception:
        return [r["tableName"] for r in spark.sql("SHOW TABLES").collect()]


expected = {name for name, *_ in results}
orphans = sorted(
    t for t in bronze_table_names()
    if t.startswith("bz_") and t not in expected
)

print()
if not orphans:
    print(f"テーブル一覧は着地ファイルと一致しています（bz_* {len(expected)} 本）。")
else:
    print(f"[整理] 対応する CSV が無いテーブルが {len(orphans)} 本あります:")
    for t in orphans:
        try:
            n = spark.table(t).count()
        except Exception:
            n = -1
        print(f"  - {t}" + (f"（{n:,} 行）" if n >= 0 else ""))

    if DROP_ORPHAN_TABLES:
        for t in orphans:
            spark.sql(f"DROP TABLE IF EXISTS {t}")
        print(f"\n{len(orphans)} 本を削除しました。")
        remain = sorted(t for t in bronze_table_names() if t.startswith("bz_"))
        print(f"ブロンズ層の bz_* テーブル: {len(remain)} 本"
              + ("" if len(remain) == len(expected) else "  ← 想定と違います"))
    else:
        print("\nDROP_ORPHAN_TABLES = False のため削除していません。")
        print("削除する場合は True にして再実行してください。")

# Files/ 側に、SOURCES に書いていないソースシステムが無いかも見ておく。
# 書き忘れると、CSV はあるのにテーブルが作られない状態になる。
try:
    landed = {e.name.rstrip("/") for e in notebookutils.fs.ls(FILES_ROOT) if e.isDir}
except Exception:
    landed = set()
untracked = sorted(landed - set(SOURCES))
if untracked:
    print()
    print(f"[WARN] SOURCES に無いフォルダーが Files/ に {len(untracked)} 件あります: "
          f"{', '.join(untracked)}")
    print("       取り込み対象なら SOURCES に追加してください。")

display(spark.createDataFrame(results, ["table_name", "row_count", "date_count", "ingest_dates"]))